# Create lookup table that links blocks with .pt files

In [ ]:
import csv
import json
from pathlib import Path


def create_tile_block_lookup(blocks_dir: str, preprocessed_dirs: list[str]):
    """Creates a lookup mapping each block subdirectory to its list of tile_id.pt files.

    Args:
        blocks_dir: Path to the main directory containing block subdirectories
        (block1-block10).
        preprocessed_dirs: List of paths to the preprocessed split directories.
    """
    blocks_path = Path(blocks_dir)

    # 1. Gather all unique tile ID numbers from preprocessed .pt files
    # Expected format: tile_idnumber.pt -> extract 'idnumber'
    available_tile_ids = set()
    for prep_dir in preprocessed_dirs:
        for pt_file in Path(prep_dir).glob("tile_*.pt"):
            # Extract the ID part (e.g. "tile_1234.pt" -> "1234")
            tile_id = pt_file.stem.split("tile_")[-1]
            available_tile_ids.add((tile_id, pt_file.name))

    tile_id_to_pt_name = dict(available_tile_ids)

    # 2. Map block directories to matching .pt filenames
    # Expected .tif format: Reference_tile_id_idnumber.tif
    block_lookup = {}

    for block_folder in sorted(blocks_path.glob("block*")):
        if block_folder.is_dir():
            matching_pt_files = []

            for tif_file in block_folder.glob("*.tif"):
                # Extract idnumber from Reference_tile_id_idnumber.tif
                tile_id = tif_file.stem.split("_")[-1]

                if tile_id in tile_id_to_pt_name:
                    matching_pt_files.append(tile_id_to_pt_name[tile_id])

            block_lookup[block_folder.name] = sorted(matching_pt_files)

    return block_lookup



# --- Configuration ---
BLOCKS_DIRECTORY = "./blocks"
PREPROCESSED_SPLITS = [
    "E:/preprocessed_train",
    "E:/preprocessed_val",
    "E:/preprocessed_test",
]

# Generate lookup dict
lookup = create_tile_block_lookup(BLOCKS_DIRECTORY, PREPROCESSED_SPLITS)

# --- Save as JSON ---
with open("block_tile_lookup.json", "w") as f:
    json.dump(lookup, f, indent=4)

# --- Save as CSV ---
with open("block_tile_lookup.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["block_id", "tile_pt_filename"])
    for block, pt_files in lookup.items():
        for pt_file in pt_files:
            writer.writerow([block, pt_file])

print("Lookup table generated successfully!")